# Projeto Completo de Data Science em R

*Machine Learning - Ciencia de Dados*

**Dataset:** German Credit Data (UCI)
**Pacotes:** ggplot2, dplyr, ranger, yardstick, rsample

**Pipeline:** Dados -> Split -> Limpeza -> Treino -> Teste

## 1. Carregar Bibliotecas

In [ ]:
# tidyverse NAO esta disponivel - usar pacotes individualmente
library(ggplot2)
library(dplyr)
library(ranger)
library(yardstick)
library(rsample)

cat("Bibliotecas carregadas!")

## 2. Carregar Dados (German Credit - UCI)

In [ ]:
# URL do dataset German Credit Data
url <- "https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.data"

# Nomes das colunas
col_names <- c("Status", "Duration", "Credit_history", "Purpose", "Amount",
               "Savings", "Employment", "Installment_rate", "Personal", "Others",
               "Residence", "Property", "Age", "Installment_plans", "Housing",
               "Existing_credits", "Job", "Liable_population", "Telephone", "Foreign", "Target")

# Carregar dados
dados <- read.table(url, col.names = col_names, stringsAsFactors = TRUE)

# Converter target: 1=bom (0), 2=mau (1) - e converter para factor!
dados$Target <- ifelse(dados$Target == 1, 0, 1)
dados$Target <- factor(dados$Target)  # IMPORTANT: factor para yardstick

cat("Dataset carregado:", nrow(dados), "observacoes", ncol(dados), "colunas\n")
str(dados$Target)

## 3. Train/Test Split (ANTES de qualquer transformacao!)

⚠️ **IMPORTANTE:** O split vem SEMPRE antes de qualquer pre-processamento!

Isso evita **DATA LEAKAGE** - quando informacao do teste vaza para o treino.

In [ ]:
set.seed(42)  # Reprodutibilidade

split <- initial_split(dados, prop = 0.8, strata = Target)
treino <- training(split)
teste <- testing(split)

cat("Treino:", nrow(treino), "observacoes\n")
cat("Teste:", nrow(teste), "observacoes\n")

# Verificar proporcao do target em cada conjunto
cat("\nProporcao no treino:\n")
table(treino$Target) %>% prop.table()

cat("\nProporcao no teste:\n")
table(teste$Target) %>% prop.table()

## 4. Limpeza (NO TREINO SOMENTE!)

⚠️ **DATA LEAKAGE:** Apenas o treino e usado para calcular estatisticas!

O teste permanece intocado até a avaliacao final.

In [ ]:
# Missing values estao representados como '?' no German Credit
treino <- treino %>%
  mutate(across(everything(), ~na_if(., "?")))

# Aplicar mediana (calculada no treino)
treino <- treino %>%
  mutate(across(where(is.numeric), ~replace_na(.x, median(.x, na.rm = TRUE))))

# Para o teste: usar mediana do TREINO
teste <- teste %>%
  mutate(across(everything(), ~na_if(., "?")))
teste <- teste %>%
  mutate(across(where(is.numeric), ~replace_na(.x, median(.x, na.rm = TRUE))))

cat("Missing values no treino:", sum(is.na(treino)), "\n")
cat("Missing values no teste:", sum(is.na(teste)), "\n")

## 5. Feature Engineering (NO TREINO SOMENTE!)

In [ ]:
# Converter variaveis categoricas para factor
treino <- treino %>%
  mutate(across(where(is.character), as.factor))

teste <- teste %>%
  mutate(across(where(is.character), as.factor))

# Verificar estrutura
glimpse(treino)

## 6. Validacao Cruzada (k-fold)

In [ ]:
# 5-fold cross-validation
cv <- vfold_cv(treino, v = 5, strata = Target)
cv

cat("\nCada linha = 1 fold\n")
cat("Na avaliacao do fold: 80% treino -> fit, 20% -> validacao\n")
cat("A media das 5 folds = score mais estavel\n")

## 7. Treino do Modelo (RandomForest com ranger)

In [ ]:
# Treinar modelo RandomForest
modelo <- ranger(
  formula = Target ~ .,
  data = treino,
  num.trees = 100,
  seed = 42
)

cat("Modelo treinado!\n")
cat("OOB Error:", modelo$prediction.error, "\n")

## 8. Feature Importance (Variaveis Mais Importantes)

In [ ]:
# Extrair importancia das variaveis
importancia <- modelo$variable.importance
importancia <- sort(importancia, decreasing = TRUE)

cat("Top 10 variaveis mais importantes:\n")
head(importancia, 10)

## 9. Avaliacao no Teste (Metricas)

⚠️ **UNICA VEZ que vemos o teste!**

Nao ajustar modelo depois desta etapa!

In [ ]:
# Fazer predicoes no teste
predicoes <- predict(modelo, data = teste)
predicoes_vec <- predicoes$predictions

# Criar dataframe para yardstick
pred_df <- data.frame(
  truth = teste$Target,
  estimate = predicoes_vec
)

cat("\n=== METRICAS NO TESTE ===\n\n")

# Accuracy
acc <- accuracy(pred_df, truth, estimate, event_level = "second")
cat("Accuracy:", acc$.estimate, "\n")

# Precision (classe positiva = 1 = mau)
prec <- precision(pred_df, truth, estimate, event_level = "second")
cat("Precision:", prec$.estimate, "\n")

# Recall
rec <- recall(pred_df, truth, estimate, event_level = "second")
cat("Recall:", rec$.estimate, "\n")

# F1 Score
f1 <- f_meas(pred_df, truth, estimate, event_level = "second")
cat("F1 Score:", f1$.estimate, "\n")

## 10. DATA LEAKAGE - Como NAO fazer

In [ ]:
# ERRADO: scale() ANTES do split = DATA LEAKAGE!
# O scaler fica com media e desvio de TODOS os dados (incluindo teste)
# Quando aplicar no treino, ja viu informacao do teste

# dados_scaled <- scale(dados)  # USA MEDIA DO TESTE!
# split <- initial_split(dados_scaled)  # Separa depois

# CERTO: Split PRIMEIRO, preprocessamento DEPOIS
# 1. Separar treino e teste primeiro
# 2. Calcular estatisticas (media, mediana, etc) SO no treino
# 3. Aplicar no treino e teste (sem ver o teste)

cat("⚠️ LEMBRETE: Split vem SEMPRE antes de qualquer transformacao!\n")
cat("scale(), mean(), median() - todos no TREINO so!\n")

## 11. Etica: COMPAS e Vies Algoritmico

### O Caso COMPAS

**COMPAS** (Correctional Offender Management Profiling for Alternative Sanctions) e um algoritmo usado nos EUA para prever reincidencia criminal.

**ProPublica descobriu (2016):**
- **Negros:** 45% de chance de ser classificados como alto risco mesmo quando NAO reincidiram
- **Brancos:** 23% de chance

Isso acontece mesmo **controlando** para historico criminal!

### Metricas de Fairness

Quando avaliamos modelos em contextos de justica, nao basta so **accuracy**:

| Metrica | Definicao | Quando Priorizar |
|---------|-----------|------------------|
| **Precision** | TP / (TP + FP) | "Das que disse risco, quantas eram?" |
| **Recall** | TP / (TP + FN) | "Das que eram risco, quantas encontrou?" |

### Discussao

1. E justo usar um modelo que erra mais para alguns grupos?
2. Quem e responsavel quando o algoritmo erra?
3. O que podemos fazer quando um modelo e biased?

---

**Exercicio:**
1. Analise: seu modelo e biased?
2. O que voce faria para mitigar?

**Entrega:** Envie o caderno pelo Moodle